# Deep Learning Descriptors — Classifier vs Metric Learning

Comparaison **à architecture identique** de deux paradigmes d'entraînement pour l'image retrieval fine-grained sur le dataset *Cars* (96 classes marque+modèle).

## Protocole

Pour chaque backbone (un CNN et un ViT), on entraîne :

1. **Mode classifier** 
2. **Mode metric learning**

Le descripteur final est donc extrait **du même point exact** dans les deux cas — c'est ce qui rend la comparaison équitable.

## Architectures choisies

| Type | Modèle                         | `d_in` natif |
|------|--------------------------------|--------------|
| CNN  | `convnext_base.fb_in22k_ft_in1k` | 1024         |
| ViT  | `vit_base_patch14_dinov2.lvd142m` | 768          |

## Analyse de réduction de dimension

Pour chaque (backbone, mode), on entraîne à `d_emb ∈ {native, native/2, native/4, native/8, native/16}` :

- **ConvNeXt-Base** : 2048 → 1024 → 512 → 256 → 128 → 64
- **DINOv2 ViT-B/14** : 2048 → 1024 → 512 → 256 → 128 → 64

À la fin, deux graphiques **mAP vs taille du descripteur** (Top-50 et Top-100) avec 4 courbes.

## 1. Imports & configuration

In [ ]:
import os
import gc
import time
import math
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from pytorch_metric_learning import losses, samplers

warnings.filterwarnings("ignore")

SEED = 123
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS_CLF    = 30
EPOCHS_METRIC = 30
SKIP_IF_EXISTS = True

print(f"Budget: EPOCHS_CLF={EPOCHS_CLF}, EPOCHS_METRIC={EPOCHS_METRIC}, SKIP_IF_EXISTS={SKIP_IF_EXISTS}")


## 2. Chargement du dataset Cars (96 classes marque+modèle)

On exclut les 15 images officielles utilisées comme queries pour que la gallery ne les contienne pas. Adapte `query_filenames` selon ton groupe (ici : groupes **impairs**).

In [ ]:
data_dir = Path("../data/raw/Cars")

query_filenames = [
    "0_1_BMW_X3_207.jpg", "0_0_BMW_Serie3Berline_74.jpg", "0_2_BMW_i8_299.jpg",
    "2_0_Volkswagen_Touareg_2822.jpg", "2_4_Volkswagen_Polo_3463.jpg", "2_9_Volkswagen_T-Roc_4209.jpg",
    "4_2_Opel_vivarofourgon_5999.jpg", "4_4_Opel_Insignatourer_6353.jpg", "4_9_Opel_zafiralife_6887.jpg",
    "6_0_Hyundai_Nexo_8282.jpg", "6_3_Hyundai_i10_8837.jpg", "6_5_Hyundai_i30_9125.jpg",
    "8_1_Ford_Puma_11276.jpg", "8_5_Ford_Explorer_11897.jpg", "8_6_Ford_Focus_11951.jpg",
]
queries_to_exclude = set(query_filenames)

all_image_paths = [f for f in sorted(data_dir.glob("*.jpg")) if f.name not in queries_to_exclude]
all_labels = []
for img_path in all_image_paths:
    parts = img_path.stem.split('_')
    all_labels.append(f"{parts[0]}_{parts[1]}")

label_encoder = LabelEncoder()
all_labels_encoded = label_encoder.fit_transform(all_labels)
NUM_CLASSES = len(label_encoder.classes_)

print(f"Images gallery : {len(all_image_paths)}")
print(f"Classes uniques : {NUM_CLASSES}")

X = np.array(all_image_paths)
y = np.array(all_labels_encoded)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
print(f"Train : {len(X_train)} | Val : {len(X_val)}")

query_paths = [data_dir / q for q in query_filenames]
query_labels = []
for q in query_filenames:
    parts = q.split('_')
    q_label_str = f"{parts[0]}_{parts[1]}"
    query_labels.append(label_encoder.transform([q_label_str])[0])
query_labels = np.array(query_labels)
print(f"Queries : {len(query_paths)}")


## 3. Dataset & transforms

In [ ]:
class CarsDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = int(self.labels[idx])
        if self.transform:
            img = self.transform(img)
        return img, label


def build_transforms(img_size=384):
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0), ratio=(0.85, 1.15)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.2)),
    ])
    val_tf = transforms.Compose([
        transforms.Resize(int(img_size * 1.10)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return train_tf, val_tf


## 4. Architectures unifiées — même backbone, même tête, flag `mode`

**Point clé pour une comparaison équitable** :

Les deux modèles (classifier et metric) partagent **exactement la même architecture** jusqu'à la couche `embedding = Linear(d_in → d_emb)`.

- En **mode classifier**, on ajoute une dernière `Linear(d_emb → num_classes)` qu'on entraîne avec CrossEntropy. À l'inférence, cette couche est **retirée** : le descripteur utilisé pour la recherche est la sortie de `Linear(d_in → d_emb)` L2-normalisée.
- En **mode metric**, on s'arrête à `Linear(d_in → d_emb)` et on L2-normalise directement. La loss SubCenter-ArcFace a ses propres weights internes (centres des classes) qui ne font **pas** partie du modèle.

Dans les deux cas, `forward()` renvoie l'embedding L2-normalisé et c'est **le même tenseur** qu'on utilise pour l'indexing.


In [ ]:
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.avg_pool2d(
            x.clamp(min=self.eps).pow(self.p),
            (x.size(-2), x.size(-1))
        ).pow(1.0 / self.p)


class ConvNeXtUnified(nn.Module):
    NATIVE_DIM = 1024
    DEFAULT_IMG_SIZE = 384

    def __init__(self, embedding_size=1024, mode="metric", num_classes=None, pretrained=True):
        super().__init__()
        assert mode in ("classifier", "metric")
        self.mode = mode
        self.embedding_size = embedding_size

        self.backbone = timm.create_model(
            'convnext_base.fb_in22k_ft_in1k',
            pretrained=pretrained,
            num_classes=0,
            global_pool='',
        )
        in_features = self.backbone.num_features
        self.pool = GeM(p=3.0)
        self.bn1 = nn.BatchNorm1d(in_features)
        self.embedding = nn.Linear(in_features, embedding_size)

        if mode == "classifier":
            assert num_classes is not None
            self.bn2 = nn.BatchNorm1d(embedding_size)
            self.classifier = nn.Linear(embedding_size, num_classes)

    def forward_features(self, x):
        f = self.backbone.forward_features(x)
        f = self.pool(f).flatten(1)
        f = self.bn1(f)
        f = self.embedding(f)
        return F.normalize(f, p=2, dim=1)

    def forward(self, x):
        if self.mode == "metric":
            return self.forward_features(x)
        f = self.backbone.forward_features(x)
        f = self.pool(f).flatten(1)
        f = self.bn1(f)
        f = self.embedding(f)
        f = self.bn2(f)
        logits = self.classifier(f)
        return logits


class DINOv2Unified(nn.Module):
    NATIVE_DIM = 768
    DEFAULT_IMG_SIZE = 224

    def __init__(self, embedding_size=768, mode="metric", num_classes=None, pretrained=True, img_size=224):
        super().__init__()
        assert mode in ("classifier", "metric")
        self.mode = mode
        self.embedding_size = embedding_size

        self.backbone = timm.create_model(
            'vit_base_patch14_dinov2.lvd142m',
            pretrained=pretrained,
            num_classes=0,
            global_pool='token',
            img_size=img_size,
            dynamic_img_size=True,
        )
        in_features = self.backbone.num_features
        self.bn1 = nn.BatchNorm1d(in_features)
        self.embedding = nn.Linear(in_features, embedding_size)

        if mode == "classifier":
            assert num_classes is not None
            self.bn2 = nn.BatchNorm1d(embedding_size)
            self.classifier = nn.Linear(embedding_size, num_classes)

    def forward_features(self, x):
        f = self.backbone(x)
        f = self.bn1(f)
        f = self.embedding(f)
        return F.normalize(f, p=2, dim=1)

    def forward(self, x):
        if self.mode == "metric":
            return self.forward_features(x)
        f = self.backbone(x)
        f = self.bn1(f)
        f = self.embedding(f)
        f = self.bn2(f)
        logits = self.classifier(f)
        return logits

if torch.cuda.is_available():
    m = ConvNeXtUnified(embedding_size=512, mode="metric").cuda()
    with torch.no_grad():
        x = torch.randn(2, 3, 384, 384).cuda()
        out = m(x)
    print(f"ConvNeXt metric — out shape: {out.shape}, norm: {out.norm(dim=1)}")
    del m; torch.cuda.empty_cache()

    m = ConvNeXtUnified(embedding_size=512, mode="classifier", num_classes=NUM_CLASSES).cuda()
    with torch.no_grad():
        out = m(x)
        emb = m.forward_features(x)
    print(f"ConvNeXt classifier — logits shape: {out.shape}, emb shape: {emb.shape}")
    del m; torch.cuda.empty_cache()


## 5. Boucles d'entraînement — classifier et metric learning

Spécificités :

| Aspect        | Classifier           | Metric learning                         |
|---------------|----------------------|-----------------------------------------|
| Loss          | `CrossEntropyLoss`   | `SubCenterArcFace(k=3, m=28.6, s=64)`   |
| Sampler       | Random shuffle       | `MPerClassSampler(m=4)`                 |
| Critère early | Val loss CE          | Val loss ArcFace                        |


In [ ]:
def cosine_warmup_scheduler(optimizer, warmup_epochs, total_epochs, min_lr_ratio=0.01):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / max(1, warmup_epochs)
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return min_lr_ratio + (1 - min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * progress))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_classifier(
    model, X_train, y_train, X_val, y_val, img_size,
    num_epochs=15, batch_size=16, accum_steps=1,
    lr=3e-4, weight_decay=1e-4, warmup_epochs=2, patience=6,
    save_path="models/clf.pth", num_workers=2,
):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    train_tf, val_tf = build_transforms(img_size=img_size)
    train_ds = CarsDataset(X_train, y_train, transform=train_tf)
    val_ds   = CarsDataset(X_val,   y_val,   transform=val_tf)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)

    model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = cosine_warmup_scheduler(optimizer, warmup_epochs, num_epochs)
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    best_val = float('inf')
    epochs_no_improve = 0
    for epoch in range(num_epochs):
        model.train()
        t0 = time.time()
        run_loss, run_correct, run_total = 0.0, 0, 0
        optimizer.zero_grad(set_to_none=True)
        for step, (imgs, labels) in enumerate(train_loader):
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=scaler is not None):
                logits = model(imgs)
                loss = criterion(logits, labels) / accum_steps
            if scaler is not None:
                scaler.scale(loss).backward()
                if (step + 1) % accum_steps == 0:
                    scaler.step(optimizer); scaler.update()
                    optimizer.zero_grad(set_to_none=True)
            else:
                loss.backward()
                if (step + 1) % accum_steps == 0:
                    optimizer.step(); optimizer.zero_grad(set_to_none=True)
            run_loss += loss.item() * accum_steps
            run_correct += (logits.argmax(1) == labels).sum().item()
            run_total += labels.size(0)
        train_loss = run_loss / max(1, len(train_loader))
        train_acc  = run_correct / max(1, run_total)

        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=scaler is not None):
                    logits = model(imgs)
                    loss = criterion(logits, labels)
                v_loss += loss.item()
                v_correct += (logits.argmax(1) == labels).sum().item()
                v_total += labels.size(0)
        val_loss = v_loss / max(1, len(val_loader))
        val_acc  = v_correct / max(1, v_total)

        scheduler.step()
        dt = time.time() - t0
        print(f"  [clf] ep {epoch+1:2d}/{num_epochs} | tr_loss {train_loss:.3f} acc {train_acc*100:5.1f}% | va_loss {val_loss:.3f} acc {val_acc*100:5.1f}% | {dt:.0f}s")

        if val_loss < best_val - 1e-4:
            best_val = val_loss
            epochs_no_improve = 0
            torch.save({'model_state': model.state_dict(), 'val_loss': val_loss,
                        'val_acc': val_acc, 'epoch': epoch + 1,
                        'mode': 'classifier', 'embedding_size': model.embedding_size},
                       save_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  [clf] early stop @ epoch {epoch+1}")
                break
    return best_val


def train_metric(
    model, X_train, y_train, X_val, y_val, img_size,
    num_epochs=20, batch_size=12, accum_steps=2,
    lr=1e-4, weight_decay=1e-4, warmup_epochs=3, patience=8,
    save_path="models/metric.pth", num_workers=2,
    m_per_class=4, sub_centers=3,
):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    train_tf, val_tf = build_transforms(img_size=img_size)
    train_ds = CarsDataset(X_train, y_train, transform=train_tf)
    val_ds   = CarsDataset(X_val,   y_val,   transform=val_tf)

    sampler = samplers.MPerClassSampler(
        labels=y_train, m=m_per_class,
        batch_size=batch_size,
        length_before_new_iter=len(X_train),
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)

    model = model.to(device)
    criterion = losses.SubCenterArcFaceLoss(
        num_classes=NUM_CLASSES,
        embedding_size=model.embedding_size,
        margin=28.6, scale=64, sub_centers=sub_centers,
    ).to(device)

    optimizer = optim.AdamW(
        [{'params': model.parameters()}, {'params': criterion.parameters()}],
        lr=lr, weight_decay=weight_decay,
    )
    scheduler = cosine_warmup_scheduler(optimizer, warmup_epochs, num_epochs)
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    best_val = float('inf')
    epochs_no_improve = 0
    for epoch in range(num_epochs):
        model.train()
        t0 = time.time()
        run_loss = 0.0
        optimizer.zero_grad(set_to_none=True)
        for step, (imgs, labels) in enumerate(train_loader):
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=scaler is not None):
                emb = model(imgs)
                loss = criterion(emb, labels) / accum_steps
            if scaler is not None:
                scaler.scale(loss).backward()
                if (step + 1) % accum_steps == 0:
                    scaler.step(optimizer); scaler.update()
                    optimizer.zero_grad(set_to_none=True)
            else:
                loss.backward()
                if (step + 1) % accum_steps == 0:
                    optimizer.step(); optimizer.zero_grad(set_to_none=True)
            run_loss += loss.item() * accum_steps
        train_loss = run_loss / max(1, len(train_loader))

        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=scaler is not None):
                    emb = model(imgs)
                    loss = criterion(emb, labels)
                v_loss += loss.item()
        val_loss = v_loss / max(1, len(val_loader))

        scheduler.step()
        dt = time.time() - t0
        print(f"  [met] ep {epoch+1:2d}/{num_epochs} | tr {train_loss:.3f} | va {val_loss:.3f} | {dt:.0f}s")

        if val_loss < best_val - 1e-4:
            best_val = val_loss
            epochs_no_improve = 0
            torch.save({'model_state': model.state_dict(), 'val_loss': val_loss,
                        'epoch': epoch + 1, 'mode': 'metric',
                        'embedding_size': model.embedding_size},
                       save_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  [met] early stop @ epoch {epoch+1}")
                break
    return best_val


## 6. Extraction des embeddings + TTA (flip horizontal)

Quel que soit le mode d'entraînement, on appelle **`forward_features`** qui renvoie le descripteur extrait au même endroit exactement.

In [ ]:
@torch.no_grad()
def extract_embeddings(model, dataloader, use_tta=True):
    model.eval()
    feats, labs = [], []
    for imgs, labels in dataloader:
        imgs = imgs.to(device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=device.type == 'cuda'):
            e1 = model.forward_features(imgs)
            if use_tta:
                e2 = model.forward_features(torch.flip(imgs, dims=[3]))
                e = F.normalize(e1 + e2, p=2, dim=1)
            else:
                e = e1
        feats.append(e.float().cpu().numpy())
        labs.append(labels.numpy() if torch.is_tensor(labels) else np.asarray(labels))
    return np.concatenate(feats), np.concatenate(labs)


def build_eval_loaders(img_size, batch_size=32, num_workers=2):
    _, val_tf = build_transforms(img_size=img_size)
    gallery_ds = CarsDataset(X, y, transform=val_tf)
    query_ds   = CarsDataset(np.array(query_paths), query_labels, transform=val_tf)
    gallery_loader = DataLoader(gallery_ds, batch_size=batch_size, shuffle=False,
                                num_workers=num_workers, pin_memory=True)
    query_loader   = DataLoader(query_ds, batch_size=batch_size, shuffle=False,
                                num_workers=num_workers, pin_memory=True)
    return gallery_loader, query_loader


## 7. Métriques de retrieval (Recall, Precision, AP, mAP, R-Prec)

In [ ]:
def average_precision(retrieved_labels, true_label, k=None):
    if k is not None:
        retrieved_labels = retrieved_labels[:k]
    hits = (retrieved_labels == true_label).astype(np.float32)
    if hits.sum() == 0:
        return 0.0
    precisions = np.cumsum(hits) / (np.arange(len(hits)) + 1)
    return float((precisions * hits).sum() / hits.sum())


def evaluate_retrieval(query_feats, gallery_feats, q_labels, g_labels, top_ks=(50, 100)):
    sim = query_feats @ gallery_feats.T
    order = np.argsort(-sim, axis=1)

    per_query = []
    for i in range(len(q_labels)):
        row = {'query_idx': i, 'true_label': int(q_labels[i])}
        ranked = g_labels[order[i]]
        n_rel = int((g_labels == q_labels[i]).sum())
        row['n_relevant'] = n_rel
        for k in top_ks:
            topk = ranked[:k]
            tp = int((topk == q_labels[i]).sum())
            row[f'R@{k}']  = tp / max(1, n_rel)
            row[f'P@{k}']  = tp / k
            row[f'AP@{k}'] = average_precision(ranked, q_labels[i], k=k)
        R = max(1, n_rel)
        row['R-Prec'] = float((ranked[:R] == q_labels[i]).sum() / R)
        per_query.append(row)

    df = pd.DataFrame(per_query)
    summary = {}
    for k in top_ks:
        summary[f'mR@{k}']  = float(df[f'R@{k}'].mean())
        summary[f'mP@{k}']  = float(df[f'P@{k}'].mean())
        summary[f'mAP@{k}'] = float(df[f'AP@{k}'].mean())
    summary['mR-Prec'] = float(df['R-Prec'].mean())
    return df, summary


def measure_index_and_search(model, img_size, batch_size):
    gallery_loader, query_loader = build_eval_loaders(img_size=img_size, batch_size=batch_size)

    t0 = time.time()
    g_feats, g_labs = extract_embeddings(model, gallery_loader, use_tta=True)
    index_time = time.time() - t0

    t0 = time.time()
    q_feats, q_labs = extract_embeddings(model, query_loader, use_tta=True)
    q_extract_time = time.time() - t0

    desc_size_MB = g_feats.nbytes / (1024 * 1024)
    
    t0 = time.time()
    sim = q_feats @ g_feats.T
    _ = np.argsort(-sim, axis=1)
    search_elapsed = time.time() - t0
    search_per_query = search_elapsed / max(1, len(q_feats)) + (q_extract_time / max(1, len(q_feats)))

    return g_feats, q_feats, g_labs, q_labs, desc_size_MB, index_time, search_per_query


## 8. Configuration des expériences

On définit les 5 tailles de descripteurs par backbone et on prépare la table des résultats.

In [ ]:
BACKBONES = {
    "ConvNeXtBase": {
        "cls": ConvNeXtUnified,
        "native_dim": 2048,
        "img_size": 384,
        "batch_clf": 16,
        "batch_met": 12,
        "accum_met": 2,
        "lr_clf": 3e-4,
        "lr_met": 1e-4,
    },
    "DINOv2_ViTB14": {
        "cls": DINOv2Unified,
        "native_dim": 2048,
        "img_size": 224,
        "batch_clf": 32,
        "batch_met": 24,
        "accum_met": 2,
        "lr_clf": 2e-4,
        "lr_met": 5e-5,
    },
}

def dim_schedule(native):
    return [native // (2 ** i) for i in range(5)]

for name, cfg in BACKBONES.items():
    print(f"{name}: native={cfg['native_dim']}, dims = {dim_schedule(cfg['native_dim'])}")


## 9. Runner : une expérience = (backbone, mode, dim)

Entraîne, extrait les features, évalue, logge une ligne dans la table de résultats.

In [ ]:
RESULTS_PATH = RESULTS_DIR / "dl_clf_vs_metric_results.json"

def load_results():
    if RESULTS_PATH.exists():
        return json.loads(RESULTS_PATH.read_text())
    return []

def save_results(results):
    RESULTS_PATH.write_text(json.dumps(results, indent=2))

results = load_results()
print(f"Résultats existants : {len(results)} lignes")


def build_model(backbone_name, mode, embedding_size):
    cfg = BACKBONES[backbone_name]
    Cls = cfg["cls"]
    kwargs = dict(embedding_size=embedding_size, mode=mode)
    if mode == "classifier":
        kwargs["num_classes"] = NUM_CLASSES
    if Cls is DINOv2Unified:
        kwargs["img_size"] = cfg["img_size"]
    return Cls(**kwargs)


def run_experiment(backbone_name, mode, embedding_size):
    cfg = BACKBONES[backbone_name]
    tag = f"{backbone_name}_{mode}_dim{embedding_size}"
    save_path = MODELS_DIR / f"{tag}.pth"

    print(f"\n{'='*70}\n  EXPERIMENT: {tag}\n{'='*70}")

    model = build_model(backbone_name, mode, embedding_size)

    need_train = not (SKIP_IF_EXISTS and save_path.exists())
    if need_train:
        if mode == "classifier":
            train_classifier(
                model, X_train, y_train, X_val, y_val,
                img_size=cfg["img_size"],
                num_epochs=EPOCHS_CLF,
                batch_size=cfg["batch_clf"],
                lr=cfg["lr_clf"],
                save_path=str(save_path),
            )
        else:
            train_metric(
                model, X_train, y_train, X_val, y_val,
                img_size=cfg["img_size"],
                num_epochs=EPOCHS_METRIC,
                batch_size=cfg["batch_met"],
                accum_steps=cfg["accum_met"],
                lr=cfg["lr_met"],
                save_path=str(save_path),
            )
    else:
        print(f"  -> checkpoint existant trouvé, skip training : {save_path.name}")

    ckpt = torch.load(save_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    model = model.to(device).eval()

    g_feats, q_feats, g_labs, q_labs, desc_size_MB, index_time, search_per_q = \
        measure_index_and_search(model, img_size=cfg["img_size"],
                                 batch_size=cfg["batch_clf"] if mode == "classifier" else cfg["batch_met"])

    df_pq, summary = evaluate_retrieval(q_feats, g_feats, q_labs, g_labs, top_ks=(50, 100))

    row = {
        "backbone": backbone_name,
        "mode": mode,
        "embedding_size": int(embedding_size),
        "desc_size_MB": float(desc_size_MB),
        "index_time_s": float(index_time),
        "search_time_per_query_s": float(search_per_q),
        "mAP@50":  summary["mAP@50"],
        "mAP@100": summary["mAP@100"],
        "mP@50":   summary["mP@50"],
        "mP@100":  summary["mP@100"],
        "mR@50":   summary["mR@50"],
        "mR@100":  summary["mR@100"],
        "mR-Prec": summary["mR-Prec"],
        "checkpoint": str(save_path),
    }
    print(f"  >>> mAP@50={row['mAP@50']*100:.2f}%  mAP@100={row['mAP@100']*100:.2f}%  "
          f"size={row['desc_size_MB']:.1f} MB  idx={row['index_time_s']:.1f}s  "
          f"search/q={row['search_time_per_query_s']*1000:.1f}ms")

    if embedding_size == BACKBONES[backbone_name]["native_dim"]:
        feats_dir = RESULTS_DIR / "features"
        feats_dir.mkdir(exist_ok=True)
        np.save(feats_dir / f"{tag}_gallery.npy", g_feats)
        np.save(feats_dir / f"{tag}_query.npy", q_feats)
        np.save(feats_dir / f"{tag}_gallery_labels.npy", g_labs)
        np.save(feats_dir / f"{tag}_query_labels.npy", q_labs)

    del model
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    return row


## 10. Lancement de toutes les expériences


In [ ]:
def experiment_key(row):
    return (row["backbone"], row["mode"], row["embedding_size"])

done_keys = {experiment_key(r) for r in results}

for backbone_name, cfg in BACKBONES.items():
    dims = dim_schedule(cfg["native_dim"])
    for mode in ("classifier", "metric"):
        for dim in dims:
            key = (backbone_name, mode, dim)
            if key in done_keys:
                print(f"[skip/already-logged] {key}")
                continue
            try:
                row = run_experiment(backbone_name, mode, dim)
                results.append(row)
                save_results(results)
            except torch.cuda.OutOfMemoryError as e:
                print(f"!! OOM on {key}: {e}")
                torch.cuda.empty_cache()
                gc.collect()
            except Exception as e:
                print(f"!! ERROR on {key}: {e}")
                raise

results_df = pd.DataFrame(results).sort_values(["backbone", "mode", "embedding_size"]).reset_index(drop=True)
results_df


## 11. Table 3 — Performance des descripteurs (format projet)

Pour chaque combinaison (backbone × mode) on prend la **meilleure taille** (celle qui maximise `mAP@50`) comme ligne de résumé.

In [ ]:
best_per_combo = (
    results_df.sort_values("mAP@50", ascending=False)
    .groupby(["backbone", "mode"], as_index=False)
    .first()
)

table3 = best_per_combo[[
    "backbone", "mode", "embedding_size",
    "index_time_s", "desc_size_MB", "search_time_per_query_s",
    "mAP@50", "mAP@100",
]].rename(columns={
    "backbone": "Descriptor",
    "mode": "Training",
    "embedding_size": "Dim",
    "index_time_s": "Indexing Time (s)",
    "desc_size_MB": "Descr. size (MB)",
    "search_time_per_query_s": "Av. Search Time/image (s)",
})
print("=== Table 3 — Meilleures performances par (backbone, mode) ===")
display(table3)
table3.to_csv(RESULTS_DIR / "table3_best_descriptors.csv", index=False)


## 12. Table 4 — Métriques par query (Top-50 / Top-100)

On refait l'évaluation détaillée sur le **meilleur modèle global** (celui qui maximise `mAP@50`).

In [ ]:
best_row = results_df.loc[results_df["mAP@50"].idxmax()]
print(f"Meilleur modèle global : {best_row['backbone']} / {best_row['mode']} / dim={best_row['embedding_size']}")
print(f"  mAP@50 = {best_row['mAP@50']*100:.2f}%, mAP@100 = {best_row['mAP@100']*100:.2f}%")

cfg = BACKBONES[best_row["backbone"]]
best_model = build_model(best_row["backbone"], best_row["mode"], int(best_row["embedding_size"]))
ckpt = torch.load(best_row["checkpoint"], map_location=device, weights_only=False)
best_model.load_state_dict(ckpt["model_state"])
best_model = best_model.to(device).eval()

_, _, _, _, _, _, _ = measure_index_and_search(
    best_model, img_size=cfg["img_size"],
    batch_size=cfg["batch_clf"] if best_row["mode"] == "classifier" else cfg["batch_met"],
)

gallery_loader, query_loader = build_eval_loaders(img_size=cfg["img_size"], batch_size=32)
g_feats, g_labs = extract_embeddings(best_model, gallery_loader, use_tta=True)
q_feats, q_labs = extract_embeddings(best_model, query_loader,   use_tta=True)
df_pq, summary = evaluate_retrieval(q_feats, g_feats, q_labs, g_labs, top_ks=(50, 100))

table4 = pd.DataFrame({
    "Query": [f"R{i+1}" for i in range(len(query_labels))],
    "R@50":   df_pq["R@50"].values,
    "R@100":  df_pq["R@100"].values,
    "P@50":   df_pq["P@50"].values,
    "P@100":  df_pq["P@100"].values,
    "AP@50":  df_pq["AP@50"].values,
    "AP@100": df_pq["AP@100"].values,
})
table4["mAP@50"]  = df_pq["AP@50"].mean()
table4["mAP@100"] = df_pq["AP@100"].mean()

print("\n=== Table 4 — Métriques par query (meilleur modèle) ===")
display(table4.style.format({c: "{:.3f}" for c in table4.columns if c != "Query"}))
table4.to_csv(RESULTS_DIR / "table4_per_query.csv", index=False)

del best_model; gc.collect()
if device.type == 'cuda': torch.cuda.empty_cache()


## 13. Graphiques — Évolution de la mAP en fonction de la taille du descripteur

Deux figures (Top-50 et Top-100), chacune avec 4 courbes : `{CNN, ViT} × {classifier, metric}`.

In [ ]:
def plot_map_vs_dim(metric_col, title):
    fig, ax = plt.subplots(figsize=(9, 5.5))
    style = {
        ("ConvNeXtBase", "classifier"):    ("o-",  "tab:blue",   "ConvNeXt-Base · classifier"),
        ("ConvNeXtBase", "metric"):        ("s--", "tab:blue",   "ConvNeXt-Base · metric learning"),
        ("DINOv2_ViTB14", "classifier"):   ("o-",  "tab:orange", "DINOv2 ViT-B/14 · classifier"),
        ("DINOv2_ViTB14", "metric"):       ("s--", "tab:orange", "DINOv2 ViT-B/14 · metric learning"),
    }
    for (backbone, mode), (marker, color, label) in style.items():
        sub = results_df[(results_df.backbone == backbone) & (results_df["mode"] == mode)]
        sub = sub.sort_values("embedding_size")
        if len(sub) == 0:
            continue
        ax.plot(sub["embedding_size"], sub[metric_col] * 100,
                marker, color=color, label=label, linewidth=2, markersize=8)
    ax.set_xscale("log", base=2)
    ax.set_xlabel("Taille du descripteur (dim)", fontsize=12)
    ax.set_ylabel(f"{metric_col} (%)", fontsize=12)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=10)

    all_dims = sorted(results_df["embedding_size"].unique())
    ax.set_xticks(all_dims)
    ax.set_xticklabels([str(d) for d in all_dims])
    plt.tight_layout()
    return fig

fig50 = plot_map_vs_dim("mAP@50", "Évolution de la mAP@50 en fonction de la taille du descripteur")
fig50.savefig(RESULTS_DIR / "map50_vs_dim.png", dpi=150, bbox_inches="tight")
plt.show()

fig100 = plot_map_vs_dim("mAP@100", "Évolution de la mAP@100 en fonction de la taille du descripteur")
fig100.savefig(RESULTS_DIR / "map100_vs_dim.png", dpi=150, bbox_inches="tight")
plt.show()


## 14. Analyse comparative synthétique

Quelques aggregations utiles pour la discussion du rapport.

In [ ]:
print("=== mAP moyenne sur toutes les tailles de descripteur ===")
agg = results_df.groupby(["backbone", "mode"]).agg(
    mAP50_mean=("mAP@50", "mean"),
    mAP100_mean=("mAP@100", "mean"),
    mAP50_max=("mAP@50", "max"),
    mAP100_max=("mAP@100", "max"),
).round(4) * 100
display(agg)

print("\n=== Écart classifier vs metric learning à dimension native ===")
for backbone, cfg in BACKBONES.items():
    native = cfg["native_dim"]
    sub = results_df[(results_df.backbone == backbone) & (results_df.embedding_size == native)]
    if len(sub) < 2:
        continue
    clf = sub[sub["mode"] == "classifier"].iloc[0]
    met = sub[sub["mode"] == "metric"].iloc[0]
    delta50  = (met["mAP@50"]  - clf["mAP@50"])  * 100
    delta100 = (met["mAP@100"] - clf["mAP@100"]) * 100
    print(f"{backbone} @ dim={native}")
    print(f"  classifier : mAP@50={clf['mAP@50']*100:.2f}%  mAP@100={clf['mAP@100']*100:.2f}%")
    print(f"  metric     : mAP@50={met['mAP@50']*100:.2f}%  mAP@100={met['mAP@100']*100:.2f}%")
    print(f"  Δ (met-clf): mAP@50={delta50:+.2f} pts  mAP@100={delta100:+.2f} pts\n")

results_df.to_csv(RESULTS_DIR / "all_results.csv", index=False)
print(f"Résultats complets : {RESULTS_DIR / 'all_results.csv'}")


## Notes pratiques

- **Checkpoints** : sauvegardés dans `../models/{backbone}_{mode}_dim{d}.pth`. Contiennent `model_state`, `val_loss`, `embedding_size`, `mode`.
- **Reprise** : tu peux interrompre la cellule 10 à tout moment. Les expériences déjà loggées dans `results/dl_clf_vs_metric_results.json` seront skippées, et les checkpoints déjà sur disque ne seront pas réentraînés (`SKIP_IF_EXISTS=True`).
- **Features Part III** : seules les features à dimension native sont sauvegardées dans `../results/features/` pour l'indexation en Part III.
- **Adapter au groupe pair** : remplacer la liste `query_filenames` en cellule 2 par celle du Table 2 de l'énoncé.
- **Équité de la comparaison** : les deux modes extraient le descripteur **au même endroit** du graphe (`forward_features` = sortie de `Linear(d_in → d_emb)` L2-normalisée). La seule différence est la loss et le sampler.
